# Motion Capture–AI VR Animation Pipeline — End-to-End Demo

**Associated manuscript:**  
He, F., Liu, X., Zhu, Z., Zhai, L., & Liu, J. (2025).  
*The Role of Motion Capture and AI in VR-Based 3D Animation Design and Production.*  
Submitted to **The Visual Computer** (Springer).  

> If you use this code, please cite the manuscript above.

This notebook walks through the complete pipeline:
1. Load raw motion capture data
2. Kalman filter smoothing (Eq. 2)
3. IK retargeting (Eq. 3)
4. CNN-LSTM facial animation (Eq. 4)
5. Latency simulation (Eq. 5)
6. Evaluation metrics

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
print('Dependencies OK')

## Step 1 — Load Raw Motion Capture Data

In [ ]:
from pipeline.acquisition.mocap_reader import load_mocap_json, normalize

data_path = '../data/sample/raw_mocap.json'
raw_data  = load_mocap_json(data_path)

print(f'Loaded {len(raw_data)} subjects')
for k, v in list(raw_data.items())[:3]:
    print(f'  {k}: shape={v.shape}  (frames × markers × 3)')

## Step 2 — Kalman Filter Smoothing (Eq. 2)

In [ ]:
from pipeline.preprocessing.kalman_smoother import kalman_smooth, compute_noise_reduction

subj = 'subject_01'
raw  = raw_data[subj]
smooth = kalman_smooth(raw, gain=0.3)

stats = compute_noise_reduction(raw, smooth)
print('Noise reduction stats:')
for k, v in stats.items():
    print(f'  {k}: {v}')

# Plot marker 0, X axis
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(raw[:, 0, 0],    alpha=0.5, label='Raw')
ax.plot(smooth[:, 0, 0], lw=2,      label='Kalman (K=0.3)')
ax.set_title('Kalman Filter — Marker 0, X axis')
ax.set_xlabel('Frame'); ax.set_ylabel('Position (m)')
ax.legend(); plt.tight_layout()
plt.savefig('../data/sample/kalman_plot.png', dpi=100)
print('Plot saved → data/sample/kalman_plot.png')

## Step 3 — IK Retargeting (Eq. 3)

In [ ]:
from pipeline.retargeting.ik_solver import retarget_sequence

# Use first 50 frames for speed in demo
result = retarget_sequence(smooth[:50], n_joints=22, n_dof=66,
                            tolerance=1e-3, max_iterations=100)

print(f"Convergence rate : {result['convergence_rate_pct']:.1f}%")
print(f"Mean error       : {result['errors'].mean()*1000:.2f} mm")
print(f"Max error        : {result['errors'].max()*1000:.2f} mm")
print(f"Mean iterations  : {result['iterations'].mean():.1f}")

## Step 4 — CNN-LSTM Facial Animation (Eq. 4)

In [ ]:
from pipeline.facial_animation.cnn_lstm_model import CNNLSTMFacialAnimator

weights_path = '../data/model_weights/cnn_lstm_facial.npz'
model = CNNLSTMFacialAnimator(weights_path=weights_path)

with open('../data/sample/facial_animation.json') as f:
    facial = json.load(f)

markers = np.array(facial['subject_01']['markers'], dtype=np.float32)  # (T,68,3)
gt_weights = np.array(facial['subject_01']['blendshape_weights'], dtype=np.float32)

pred_weights = model.predict(markers[:100], reset_state=True)
mse = float(np.mean((pred_weights - gt_weights[:100])**2))
print(f'Blendshape prediction MSE: {mse:.4f}')
print(f'Predicted shape: {pred_weights.shape}  (frames × blendshapes)')
print(f'Weight range: [{pred_weights.min():.3f}, {pred_weights.max():.3f}] (expect [0,1])')

## Step 5 — Latency Simulation (Eq. 5)

In [ ]:
from pipeline.rendering.latency_monitor import simulate_latency_report

report = simulate_latency_report(n_frames=300)
print('VR Latency Report:')
for k, v in report.items():
    print(f'  {k}: {v}')

## Step 6 — Full Evaluation Report

In [ ]:
from evaluation.metrics import load_and_report
_ = load_and_report('../data/sample/evaluation_metrics.json')

---
## Citation

```bibtex
@article{he2025mocap_vr,
  author  = {He, Fang and Liu, Xun and Zhu, Zhaoye and Zhai, Lanru and Liu, Jiaxin},
  title   = {The Role of Motion Capture and AI in VR-Based 3D Animation Design and Production},
  journal = {The Visual Computer},
  year    = {2025},
  note    = {Submitted}
}
```